# **Notebook 3 - Basket Analysis**

Afonso Fernandes 20241710, Lourenço Lima 20241711, Lucas Casimiro 20241796

## Imports

In [1]:
import os
import sys
import warnings
from pathlib import Path

def _find_project_root(start, marker="requirements.txt"):
    path = Path(start).resolve()
    for candidate in [path] + list(path.parents):
        if (candidate / marker).exists():
            return str(candidate)
    raise RuntimeError(f"Could not find project root (marker={marker!r}, searched from {start})")

PROJECT_ROOT = _find_project_root(os.path.abspath("."))
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import seaborn as sns
import math
from sklearn.preprocessing import RobustScaler


from functions.eda import *
from functions.preprocessing import *
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# **1. Merging data**

In [3]:
customer_basket = pd.read_csv('data/customer_basket.csv')
customer = pd.read_csv('data/ci_clustered.csv')

In [4]:
customer_merged = pd.merge(
    customer_basket, 
    customer[['customer_id', 'final_cluster_label']], 
    on='customer_id', 
    how='inner')
customer_merged

,invoice_id,list_of_goods,customer_id,final_cluster_label
0,3700630,"['chicken', 'rice', 'pepper', 'whole wheat ric...",12912,Loyal core spenders
1,10242376,"['low fat yogurt', 'tomatoes', 'pepper', 'aspa...",22853,Bargain hunters
2,91550,"['cake', 'tomatoes', 'pancakes', 'iPad', 'fina...",19,Bargain hunters
3,3137503,"['cereals', 'megaman zero', 'final fantasy XIX...",10995,Big families (big spenders)
4,7165061,"['rice', 'frozen smoothie', 'black tea', 'tea'...",27807,Big families (big spenders)
...,...,...,...,...
98699,7685930,"['yams', 'airpods', 'shrimp']",21357,Loyal core spenders
98700,2998960,"['cake', 'chocolate bread', 'cologne', 'beer',...",7817,Tech enthusiasts
98701,4409376,"['chicken', 'frozen smoothie', 'milk', 'body s...",35497,Loyal core spenders
98702,3650163,"['metroid fusion', 'cat food', 'melons', 'ring...",13412,Vegans


In [6]:
cluster_groups = customer_merged.groupby('final_cluster_label')
cluster_groups.head(5)

,invoice_id,list_of_goods,customer_id,final_cluster_label
0,3700630,"['chicken', 'rice', 'pepper', 'whole wheat ric...",12912,Loyal core spenders
1,10242376,"['low fat yogurt', 'tomatoes', 'pepper', 'aspa...",22853,Bargain hunters
2,91550,"['cake', 'tomatoes', 'pancakes', 'iPad', 'fina...",19,Bargain hunters
3,3137503,"['cereals', 'megaman zero', 'final fantasy XIX...",10995,Big families (big spenders)
4,7165061,"['rice', 'frozen smoothie', 'black tea', 'tea'...",27807,Big families (big spenders)
5,9814176,"['cottage cheese', 'samsung galaxy 10', 'straw...",36382,Vegans
6,4504266,"['rice', 'white wine', 'turkey', 'shallot', 'b...",27921,Vegans
7,7663691,"['antioxydant juice', 'soda', 'shower gel', 'c...",22128,Tech enthusiasts
8,11046498,"['barbecue sauce', 'whole wheat rice', 'mint',...",14948,Karens
9,7253615,"['ratchet & clank 3', 'rice', 'salmon', 'whole...",6169,Loyal core spenders


Note: some baskets were lost due to outlier removal.


**Temos q ver como fazemos com isto, se so aceitamos ou corremos os notebooks todos com os outliers mm**